In [1]:
!pip install sacremoses
import warnings
warnings.filterwarnings("ignore")
from argparse import ArgumentParser
import itertools
from nltk.tokenize import TweetTokenizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 22.6 MB/s eta 0:00:00


In [2]:
from transformers import BertConfig, BertTokenizer, BertForSequenceClassification, BertForPreTraining, BertModel

def load_model(args):
    if 'indobenchmark' in args['model_checkpoint']:
        # Prepare config & tokenizer
        vocab_path, config_path = None, None
        tokenizer = BertTokenizer.from_pretrained(args['model_checkpoint'])
        config = BertConfig.from_pretrained(args['model_checkpoint'])
        if type(args['num_labels']) == list:
            config.num_labels = max(args['num_labels'])
            config.num_labels_list = args['num_labels']
        else:
            config.num_labels = args['num_labels']

        print(f"Model config.num_labels before model instantiation: {config.num_labels}") # Added print statement

        # Instantiate model
        model_class = None
        if 'sequence_classification' == args['task']:
            model_class = AlbertForSequenceClassification if 'lite' in args['model_checkpoint'] else BertForSequenceClassification
        model = model_class.from_pretrained(args['model_checkpoint'], config=config)
    return model, tokenizer, vocab_path, config_path

In [3]:
import torch

# Forward function for sequence classification
def forward_sequence_classification(model, batch_data, i2w, is_test=False, device='cpu', **kwargs):
    # Unpack batch data
    if len(batch_data) == 3:
        (subword_batch, mask_batch, label_batch) = batch_data
        token_type_batch = None
    elif len(batch_data) == 4:
        (subword_batch, mask_batch, token_type_batch, label_batch) = batch_data

    # Prepare input & label
    subword_batch = torch.LongTensor(subword_batch)
    mask_batch = torch.FloatTensor(mask_batch)
    token_type_batch = torch.LongTensor(token_type_batch) if token_type_batch is not None else None
    label_batch = torch.LongTensor(label_batch)

    if device == "cuda":
        subword_batch = subword_batch.cuda()
        mask_batch = mask_batch.cuda()
        token_type_batch = token_type_batch.cuda() if token_type_batch is not None else None
        label_batch = label_batch.cuda()

    # Forward model
    outputs = model(subword_batch, attention_mask=mask_batch, token_type_ids=token_type_batch, labels=label_batch)
    loss, logits = outputs[:2]

    # generate prediction & label list
    list_hyp = []
    list_label = []
    hyp = torch.topk(logits, 1)[1]
    for j in range(len(hyp)):
        list_hyp.append(i2w[hyp[j].item()])
        list_label.append(i2w[label_batch[j].item()])

    return loss, list_hyp, list_label

In [4]:
import numpy as np
import pandas as pd
import string
import re
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class TopHuDataset(Dataset):
    # Static constant variable
    LABEL2INDEX = {
    'Pidana': 0,
    'Ketenagakerjaan': 1,
    'Keluarga': 2,
    'Kenegaraan': 3,
    'Bisnis': 4,
    'Perdata': 5,
    'Pertanahan & Properti': 6,
    'Kekayaan Intelektual': 7,
    'Ilmu Hukum': 8,
    'Perlindungan Konsumen': 9,
    'Profesi Hukum': 10,
    'Teknologi': 11,
    'Hak Asasi Manusia': 12
    }
    INDEX2LABEL = {v: k for k, v in LABEL2INDEX.items()}
    NUM_LABELS = 13


    def load_dataset(self, path):
        # Load dataset
        dataset = pd.read_csv(path)
        print("Unique labels in dataset:", dataset['label'].unique())
        # Keep only rows where 'label' is in LABEL2INDEX keys
        valid_labels = list(self.LABEL2INDEX.keys())
        dataset = dataset[dataset['label'].isin(valid_labels)].reset_index(drop=True) # Filter out rows with labels not in LABEL2INDEX and reset index
        dataset['label'] = dataset['label'].apply(lambda sen: self.LABEL2INDEX[sen])
        return dataset

    def __init__(self, dataset_path, tokenizer, no_special_token=False, *args, **kwargs):
        self.data = self.load_dataset(dataset_path)
        self.tokenizer = tokenizer
        self.no_special_token = no_special_token

    def __getitem__(self, index):
        tweet, label = self.data.loc[index,'question'], self.data.loc[index,'label']
        subwords = self.tokenizer.encode(tweet, add_special_tokens=not self.no_special_token)
        return np.array(subwords), np.array(label), tweet

    def __len__(self):
        return len(self.data)

class TopHuDataLoader(DataLoader):
    def __init__(self, max_seq_len=128, *args, **kwargs):
        super(TopHuDataLoader, self).__init__(*args, **kwargs)
        self.collate_fn = self._collate_fn
        self.max_seq_len = max_seq_len

    def _collate_fn(self, batch):
        batch_size = len(batch)
        max_seq_len = max(map(lambda x: len(x[0]), batch))
        max_seq_len = min(self.max_seq_len, max_seq_len)

        subword_batch = np.zeros((batch_size, max_seq_len), dtype=np.int64)
        mask_batch = np.zeros((batch_size, max_seq_len), dtype=np.float32)
        label_batch = np.full((batch_size,), -100, dtype=np.int64) # Changed shape to (batch_size,)

        seq_list = []
        for i, (subwords, label, raw_seq) in enumerate(batch):
            subwords = subwords[:max_seq_len]
            subword_batch[i,:len(subwords)] = subwords
            mask_batch[i,:len(subwords)] = 1
            label_batch[i] = label # Assign the single label value

            seq_list.append(raw_seq)

        return subword_batch, mask_batch, label_batch, seq_list

In [5]:
w2i, i2w = TopHuDataset.LABEL2INDEX, TopHuDataset.INDEX2LABEL
print(w2i)
print(i2w)

{'Pidana': 0, 'Ketenagakerjaan': 1, 'Keluarga': 2, 'Kenegaraan': 3, 'Bisnis': 4, 'Perdata': 5, 'Pertanahan & Properti': 6, 'Kekayaan Intelektual': 7, 'Ilmu Hukum': 8, 'Perlindungan Konsumen': 9, 'Profesi Hukum': 10, 'Teknologi': 11, 'Hak Asasi Manusia': 12}
{0: 'Pidana', 1: 'Ketenagakerjaan', 2: 'Keluarga', 3: 'Kenegaraan', 4: 'Bisnis', 5: 'Perdata', 6: 'Pertanahan & Properti', 7: 'Kekayaan Intelektual', 8: 'Ilmu Hukum', 9: 'Perlindungan Konsumen', 10: 'Profesi Hukum', 11: 'Teknologi', 12: 'Hak Asasi Manusia'}


**Train 1**



In [ ]:
import os
import shutil
import random
from torch import optim
from tqdm import tqdm
# from transformers import AdamW
from torch.optim import AdamW
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import argparse

# Set seed for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

# Function to check if running in a notebook environment
def in_notebook():
    try:
        get_ipython
        return True
    except NameError:
        return False

# Metrics calculation function
def topik_hukum_metrics_fn(list_hyp, list_label):
    metrics = {}
    metrics["ACC"] = accuracy_score(list_label, list_hyp)
    metrics["F1"] = f1_score(list_label, list_hyp, average='macro')
    metrics["REC"] = recall_score(list_label, list_hyp, average='macro')
    metrics["PRE"] = precision_score(list_label, list_hyp, average='macro')
    return metrics

# Functions for logging and training
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def metrics_to_string(metric_dict):
    return ' '.join(f"{key}:{value:.5f}" for key, value in metric_dict.items())

class MetricsTracker:
    def __init__(self):
        self.train_losses = []
        self.valid_losses = []
        self.train_metrics = []
        self.valid_metrics = []
        self.best_epoch = 0
        self.best_val_loss = float('inf')  # Changed from best_val_metric

        # Define colors
        self.train_color = '#3498db'  # Light blue
        self.valid_color = '#ff7f0e'  # Orange
        self.best_line_color = '#e74c3c'  # Red

    def update_train(self, loss, metrics):
        self.train_losses.append(loss)
        self.train_metrics.append(metrics)

    def update_valid(self, loss, metrics, epoch):  # Removed val_metric parameter
        self.valid_losses.append(loss)
        self.valid_metrics.append(metrics)

        # Track best epoch based on validation loss
        if loss < self.best_val_loss:
            self.best_val_loss = loss
            self.best_epoch = epoch

    def plot_metrics(self, save_dir):
        epochs = range(1, len(self.train_losses) + 1)

        # Set style
        plt.style.use('seaborn-v0_8-whitegrid')

        # Create a figure with two subplots
        fig, axes = plt.subplots(1, 2, figsize=(28, 14))

        # Plot F1 Score
        train_f1 = [m['F1'] for m in self.train_metrics]  # Removed * 100
        valid_f1 = [m['F1'] for m in self.valid_metrics]  # Removed * 100

        ax1 = axes[0]
        ax1.plot(epochs, train_f1, '-o', color=self.train_color, label='Training F1-SCORE', linewidth=2, alpha=0.8)
        ax1.plot(epochs, valid_f1, '-o', color=self.valid_color, label='Validation F1-SCORE', linewidth=2, alpha=0.8)
        ax1.axvline(x=self.best_epoch + 1, color=self.best_line_color, linestyle='--', label=f'Best Model (Epoch {self.best_epoch + 1})', alpha=0.8)
        ax1.set_ylim(0, 1)
        ax1.set_yticks(np.arange(0, 1.1, 0.1))  # 0.0, 0.1, ..., 1.0
        ax1.set_title('Training and Validation F1-SCORE')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('F1-SCORE (%)')
        ax1.legend(loc='lower right')
        ax1.grid(True)

        # Plot Loss
        ax2 = axes[1]
        ax2.plot(epochs, self.train_losses, '-o', color=self.train_color, label='Training Loss', linewidth=2, alpha=0.8)
        ax2.plot(epochs, self.valid_losses, '-o', color=self.valid_color, label='Validation Loss', linewidth=2, alpha=0.8)
        ax2.axvline(x=self.best_epoch + 1, color=self.best_line_color, linestyle='--', label=f'Best Model (Epoch {self.best_epoch + 1})', alpha=0.8)
        ax2.set_ylim(0, max(max(self.train_losses), max(self.valid_losses)) * 1.1)  # Dynamic y-axis limit
        ax2.set_title('Training and Validation Loss')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend(loc='upper right')
        ax2.grid(True)

        plt.tight_layout()
        combined_path = os.path.join(save_dir, 'combined_metrics.png')
        plt.savefig(combined_path, dpi=300)
        plt.close()


    def save_metrics_to_csv(self, save_dir):
        data = []
        for epoch in range(len(self.train_losses)):
            epoch_data = {
                'Epoch': epoch + 1,
                'Train Loss': self.train_losses[epoch],
                'Valid Loss': self.valid_losses[epoch],
                'Train ACC': self.train_metrics[epoch]['ACC'],
                'Valid ACC': self.valid_metrics[epoch]['ACC'],
                'Train Precision': self.train_metrics[epoch]['PRE'],
                'Valid Precision': self.valid_metrics[epoch]['PRE'],
                'Train Recall': self.train_metrics[epoch]['REC'],
                'Valid Recall': self.valid_metrics[epoch]['REC'],
                'Train F1': self.train_metrics[epoch]['F1'],
                'Valid F1': self.valid_metrics[epoch]['F1'],
                'Is Best Epoch': epoch == self.best_epoch
            }
            data.append(epoch_data)

        df = pd.DataFrame(data)
        df.to_csv(os.path.join(save_dir, 'training_metrics.csv'), index=False)

# Evaluation function
def evaluate(model, data_loader, forward_fn, metrics_fn, i2w, args, is_test=False):
    model.eval()
    total_loss = 0
    list_hyp, list_label, list_seq = [], [], []
    pbar = tqdm(data_loader, total=len(data_loader))

    with torch.no_grad():
        for i, batch_data in enumerate(pbar):
            batch_seq = batch_data[-1]
            loss, batch_hyp, batch_label = forward_fn(model, batch_data[:-1], i2w=i2w, device=args['device'])
            total_loss += loss.item()
            list_hyp.extend(batch_hyp)
            list_label.extend(batch_label)
            list_seq.extend(batch_seq)
            metrics = metrics_fn(list_hyp, list_label)

            desc = "TEST LOSS:" if is_test else "VALID LOSS:"
            pbar.set_description(f"{desc}{total_loss/(i+1):.4f} {metrics_to_string(metrics)}")

    if is_test:
        return total_loss, metrics, list_hyp, list_label, list_seq
    return total_loss, metrics

# Training function
def train(model, train_loader, valid_loader, optimizer, forward_fn, metrics_fn, valid_criterion, i2w, args):
    best_val_loss = float('inf')  # Changed from best_val_metric
    count_stop = 0
    metrics_tracker = MetricsTracker()

    for epoch in range(args['n_epochs']):
        model.train()
        total_train_loss = 0
        list_hyp, list_label = [], []

        train_pbar = tqdm(train_loader, total=len(train_loader))
        for i, batch_data in enumerate(train_pbar):
            loss, batch_hyp, batch_label = forward_fn(model, batch_data[:-1], i2w=i2w, device=args['device'])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            list_hyp.extend(batch_hyp)
            list_label.extend(batch_label)

            train_pbar.set_description(f"(Epoch {epoch+1}) TRAIN LOSS:{total_train_loss/(i+1):.4f} LR:{get_lr(optimizer):.8f}")

        # Calculate training metrics
        train_loss = total_train_loss / len(train_loader)
        train_metrics = metrics_fn(list_hyp, list_label)
        metrics_tracker.update_train(train_loss, train_metrics)

        print(f"(Epoch {epoch+1}) TRAIN LOSS:{train_loss:.4f} {metrics_to_string(train_metrics)} LR:{get_lr(optimizer):.8f}")

        # Validation step
        if (epoch+1) % args['evaluate_every'] == 0:
            val_loss, val_metrics = evaluate(model, valid_loader, forward_fn, metrics_fn, i2w, args, is_test=False)
            val_loss_avg = val_loss / len(valid_loader)

            # Update metrics tracker with current validation results
            metrics_tracker.update_valid(val_loss_avg, val_metrics, epoch)

            print(f"(Epoch {epoch+1}) VALID LOSS:{val_loss_avg:.4f} {metrics_to_string(val_metrics)}")

            # Early stopping based on validation loss
            if val_loss_avg < best_val_loss:
                best_val_loss = val_loss_avg
                torch.save(model.state_dict(), os.path.join(args['model_dir'], "best_model.th"))
                count_stop = 0
            else:
                count_stop += 1
                if count_stop == args['early_stop']:
                    print(f"\nEarly stopping triggered after {epoch+1} epochs!")
                    break

    # Generate plots and save metrics
    metrics_tracker.plot_metrics(args['model_dir'])
    metrics_tracker.save_metrics_to_csv(args['model_dir'])

    return metrics_tracker

# Main execution
if __name__ == "__main__":
    if in_notebook():
        args = {
            "experiment_name": "test_experiment",
            "model_dir": "./model_dir",
            "task": "sequence_classification",
            "model_checkpoint": "indobenchmark/indobert-base-p1",
            "dataset_class": TopHuDataset,
            "train_set_path": "/content/drive/MyDrive/Datasets/train_data.csv",
            "valid_set_path": "/content/drive/MyDrive/Datasets/valid_data.csv",
            "test_set_path": "/content/drive/MyDrive/Datasets/test_data.csv",
            "dataloader_class": TopHuDataLoader,
            "num_labels": TopHuDataset.NUM_LABELS,
            "forward_fn": forward_sequence_classification,
            "metrics_fn": topik_hukum_metrics_fn,
            "word_tokenizer_class": TweetTokenizer,
            "valid_criterion": "F1",
            "lr": 0.00003,
            "max_seq_len": 128,
            "train_batch_size": 32 ,
            "valid_batch_size": 32 ,
            "test_batch_size": 32 ,
            "n_epochs": 10,
            "early_stop": 3,
            "device": "cuda" if torch.cuda.is_available() else "cpu",
            "fp16": False,
            "force": True,
            "no_special_token": False,
            "lower": True,
            "seed": 42,
            "evaluate_every": 1
        }
    else:
        args = vars(get_args())

    # Set seed and prepare model directory
    set_seed(args['seed'])
    if not os.path.exists(args["model_dir"]):
        os.makedirs(args["model_dir"], exist_ok=True)
    elif args['force']:
        print(f'Overwriting model directory `{args["model_dir"]}`')
    else:
        raise Exception(f"Model directory `{args['model_dir']}` already exists. Use --force to overwrite.")

    # Load your model and data loaders here
    model, tokenizer, vocab_path, config_path = load_model(args)
    model = model.to(args['device'])
    optimizer = optim.Adam(model.parameters(), lr=args['lr'])

    train_dataset_path = args['train_set_path']
    train_dataset = args['dataset_class'](train_dataset_path, tokenizer, lowercase=args["lower"], no_special_token=args['no_special_token'])
    train_loader = args['dataloader_class'](dataset=train_dataset, max_seq_len=args['max_seq_len'], batch_size=args['train_batch_size'], num_workers=16, shuffle=True)

    valid_dataset_path = args['valid_set_path']
    valid_dataset = args['dataset_class'](valid_dataset_path, tokenizer, lowercase=args["lower"], no_special_token=args['no_special_token'])
    valid_loader = args['dataloader_class'](dataset=valid_dataset, max_seq_len=args['max_seq_len'], batch_size=args['valid_batch_size'], num_workers=16, shuffle=True)

    test_dataset_path = args['test_set_path']
    test_dataset = args['dataset_class'](test_dataset_path, tokenizer, lowercase=args["lower"], no_special_token=args['no_special_token'])
    test_loader = args['dataloader_class'](dataset=test_dataset, max_seq_len=args['max_seq_len'], batch_size=args['test_batch_size'], num_workers=16, shuffle=True)

    # Train model
    metrics_tracker = train(model, train_loader, valid_loader, optimizer, args['forward_fn'],
                          args['metrics_fn'], args['valid_criterion'], i2w, args)

    # Evaluation phase
    print("=========== EVALUATION PHASE ===========")
    model.load_state_dict(torch.load(os.path.join(args["model_dir"], "best_model.th")))
    test_loss, test_metrics, test_hyp, test_label, test_seq = evaluate(model, test_loader,
                                                                      args['forward_fn'], args['metrics_fn'],
                                                                      i2w, args, is_test=True)

    # Save evaluation results
    print("\nTest Metrics:")
    print(metrics_to_string(test_metrics))

    # Save results to files
    result_df = pd.DataFrame({
        'seq': test_seq,
        'hyp': test_hyp,
        'label': test_label
    })
    result_df.to_csv(os.path.join(args["model_dir"], "prediction_result.csv"), index=False)

    # Save final metrics
    final_metrics = {
        'test_loss': test_loss / len(test_loader),
        **test_metrics
    }
    metric_df = pd.DataFrame([final_metrics])
    metric_df.to_csv(os.path.join(args["model_dir"], "evaluation_result.csv"), index=False)

    # Save model configuration
    config = {
        'experiment_name': args['experiment_name'],
        'model_dir': args['model_dir'],
        'learning_rate': args['lr'],
        'max_seq_len': args['max_seq_len'],
        'batch_size': args['train_batch_size'],
        'n_epochs': args['n_epochs'],
        'early_stop': args['early_stop'],
        'seed': args['seed'],
        'device': args['device'],
        'lower': args['lower'],
        'no_special_token': args['no_special_token']
    }

    config_df = pd.DataFrame([config])
    config_df.to_csv(os.path.join(args["model_dir"], "config.csv"), index=False)

    print("\nTraining completed! Results saved in:", args["model_dir"])
    print("Files saved:")
    print("- best_model.th: Best model weights")
    print("- prediction_result.csv: Test set predictions")
    print("- evaluation_result.csv: Final test metrics")
    print("- training_metrics.xlsx: Per-epoch training and validation metrics")
    print("- config.csv: Model configuration")
    print("- loss_plot.png: Training and validation loss curves")
    print("- accuracy_plot.png: Training and validation accuracy curves")

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

Model config.num_labels before model instantiation: 13


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

... (7 outputs left out)

(Epoch 1) TRAIN LOSS:1.1119 ACC:0.66012 F1:0.55384 REC:0.52687 PRE:0.65438 LR:0.00003000


VALID LOSS:0.8370 ACC:0.74202 F1:0.66387 REC:0.65157 PRE:0.70021: 100%|██████████| 25/25 [04:51<00:00, 11.64s/it]


(Epoch 1) VALID LOSS:0.8370 ACC:0.74202 F1:0.66387 REC:0.65157 PRE:0.70021


  0%|          | 0/196 [00:00<?, ?it/s]

In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizer

w2i, i2w = TopHuDataset.LABEL2INDEX, TopHuDataset.INDEX2LABEL
print(w2i)
print(i2w)

# Load the model and tokenizer
def load_model(model_path, num_labels):
    # Instantiate the model with the correct number of labels
    model = BertForSequenceClassification.from_pretrained('indobenchmark/indobert-base-p1', num_labels=num_labels)
    # Load the state dictionary
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')), strict=True) # Changed strict to True
    model.eval()  # Set to evaluation mode
    return model

# Define the path to the model directory and the model file from the training run
MODEL_DIR = './model_dir' # Changed to the correct model directory
MODEL_PATH = f"{MODEL_DIR}/best_model.th"
NUM_LABELS = TopHuDataset.NUM_LABELS  # Use the number of labels from the dataset class

# Load the model and tokenizer
model = load_model(MODEL_PATH, NUM_LABELS)
tokenizer = BertTokenizer.from_pretrained('indobenchmark/indobert-base-p1')

# Ensure the model is on the correct device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Prediction function
def predict_sentence(model, tokenizer, sentence, max_seq_len=128):
    model.eval()

    # Tokenize and encode the input sentence
    inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True, max_length=max_seq_len)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)  # Get probabilities
        predicted_class = torch.argmax(probabilities, dim=1).item()

    # Convert predicted class index to label
    predicted_label = i2w[predicted_class]  # Use i2w to get the label

    return predicted_class, predicted_label, probabilities[0].cpu().numpy()  # Return probabilities as well

# Example usage
sentence = "Ada yang ingin saya tanyakan, bagaimana pembuatan kk poligami jika suami punya dua istri dengan pernikahan yang sah? Apakah nama suami dapat tercantum di dua KK yang berbeda? Mohon solusinya. Terima kasih."
predicted_class, predicted_label, probabilities = predict_sentence(model, tokenizer, sentence)

print(f"Predicted class index: {predicted_class}, Predicted label: {predicted_label}")
print(f"Probabilities: {probabilities}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Fungsi untuk evaluasi model menggunakan topik_hukum_metrics_fn
def evaluate_model(test_data, true_labels):
    """
    Evaluasi model dengan menggunakan topik_hukum_metrics_fn.

    Args:
        test_data (list): Data uji berupa teks (questions).
        true_labels (list): Label ground truth.

    Returns:
        dict: Hasil evaluasi ('ACC', 'F1', 'REC', 'PRE').
        DataFrame: Sampel hasil prediksi.
    """
    # Kumpulkan semua prediksi
    predicted_labels = []
    all_probabilities = []

    for question in test_data:
        _, predicted_label, probabilities = predict_sentence(model, tokenizer, question)
        predicted_labels.append(predicted_label)
        all_probabilities.append(probabilities)

    # Hitung metrik evaluasi menggunakan fungsi topik_hukum_metrics_fn
    metrics = topik_hukum_metrics_fn(predicted_labels, true_labels)

    unique_labels = sorted(list(set(true_labels)))
    conf_matrix = confusion_matrix(true_labels, predicted_labels, labels=unique_labels)

    # Ambil sampel 10 prediksi untuk verifikasi
    df = pd.DataFrame({
        'label': true_labels,
        'question': test_data,
        'predicted_label': predicted_labels,
        'probabilities': all_probabilities
    })
    sample_df = df.sample(10, random_state=42)

    return metrics, sample_df, conf_matrix, unique_labels, predicted_labels

def plot_confusion_matrix(conf_matrix, labels):
    """
    Plot confusion matrix menggunakan seaborn.

    Args:
        conf_matrix (ndarray): Confusion matrix.
        labels (list): Label untuk setiap kelas.
    """
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix,
                annot=True,
                fmt='d',
                cmap='Blues',
                xticklabels=labels,
                yticklabels=labels)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()

# Load test data
test_data_path = '/content/drive/MyDrive/Datasets/test_data.csv'
test_df = pd.read_csv(test_data_path)

# Extract questions and true labels
test_data = test_df['question'].tolist()
true_labels = test_df['label'].tolist()

# Evaluasi model
metrics, sample_df, conf_matrix, unique_labels, predicted_labels = evaluate_model(test_data, true_labels)

# Tampilkan hasil evaluasi
print("Evaluation Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value:.5f}")

print("\nConfusion Matrix:")
print(pd.DataFrame(conf_matrix,
                  index=unique_labels,
                  columns=unique_labels))

# Plot confusion matrix
plot_confusion_matrix(conf_matrix, unique_labels)

# Tambahkan laporan klasifikasi
print("\nClassification Report:")
class_report = classification_report(true_labels, predicted_labels, target_names=unique_labels)
print(class_report)

# Simpan hasil evaluasi ke dalam file Excel
with pd.ExcelWriter('evaluation_results.xlsx') as writer:
    # Simpan metrik evaluasi
    metrics_df = pd.DataFrame({
        'Metric': list(metrics.keys()),
        'Value': list(metrics.values())
    })
    metrics_df.to_excel(writer, sheet_name='Metrics', index=False)

    # Simpan confusion matrix
    conf_matrix_df = pd.DataFrame(conf_matrix,
                                index=unique_labels,
                                columns=unique_labels)
    conf_matrix_df.to_excel(writer, sheet_name='Confusion Matrix', index=True)

    # Simpan sampel prediksi
    sample_df.to_excel(writer, sheet_name='Sample Predictions', index=False)

    # Simpan laporan klasifikasi
    class_report_df = pd.DataFrame(classification_report(true_labels, predicted_labels, target_names=unique_labels, output_dict=True)).transpose()
    class_report_df.to_excel(writer, sheet_name='Classification Report')

print("Results saved to 'evaluation_results.xlsx'.")

In [ ]:
from transformers import AutoTokenizer

# ================================
# Load tokenizer IndoBERT
# ================================
tokenizer = AutoTokenizer.from_pretrained(
    "indobenchmark/indobert-base-p1",
    use_fast=False
)

# ================================
# Kalimat contoh
# ================================
text = "bisakah seseorang dapat dipidana jika menyebarkan informasi yang belum terbukti kebenarannya di media sosial"

print("Kalimat asli:")
print(text)
print()


# ================================
# 1. WordPiece Tokenizer
# ================================
tokens = tokenizer.tokenize(text)

print("1. Hasil WordPiece Tokenizer:")
print(tokens)
print()


# ================================
# 2. Tambah token khusus [CLS] dan [SEP]
# ================================
tokens_special = ["[CLS]"] + tokens + ["[SEP]"]

print("2. Setelah ditambah [CLS] dan [SEP]:")
print(tokens_special)
print()


# ================================
# 3. Konversi token ke ID
# ================================
token_ids = tokenizer.convert_tokens_to_ids(tokens_special)

print("3. Token ID:")
print(token_ids)
print()
